# Smart Traffic Light Traffic-Police DQN Training (Kaggle)

This notebook trains a stronger traffic-light policy from the `dev-truong` branch of `truongNgn/smart-traffic-light-system`.

The action space matches a real traffic officer at a Vietnamese four-way intersection:

- `Action 0 = EAST_WEST`: both opposite directions on the East-West road get green together.
- `Action 1 = NORTH_SOUTH`: both opposite directions on the North-South road get green together.

Each selected green phase is held for 10 seconds, phase switches still insert 2s yellow + 2s all-red, and the agent uses Double DQN targets with gradient clipping. The final cell creates `dqn_eval_best.pt`, selected by deterministic evaluation metrics rather than raw training reward.


## 1. Install SUMO and clone the repo

In [ ]:
!pip install -q eclipse-sumo traci sumolib libsumo

In [ ]:
!git clone --branch dev-truong --single-branch https://github.com/truongNgn/smart-traffic-light-system.git
%cd smart-traffic-light-system


## 1.1. Verify 2-phase traffic-police logic

`dev-truong` is the default branch for this project. This cell verifies that Kaggle cloned a commit with the improved 2-phase action space before spending time training.


In [ ]:
# Verify the cloned dev-truong code uses the traffic-police-style 2-phase action space.
from common.constants import DQN_OUTPUT_SIZE, NUM_ACTIONS, PHASE_DIRECTIONS, PhaseAction
from rl.env.traffic_env import SumoTrafficEnv

assert NUM_ACTIONS == 2, f"Expected 2 phase actions, got {NUM_ACTIONS}"
assert DQN_OUTPUT_SIZE == 2, f"Expected DQN output size 2, got {DQN_OUTPUT_SIZE}"
assert set(PhaseAction) == {PhaseAction.EAST_WEST, PhaseAction.NORTH_SOUTH}
assert len(PHASE_DIRECTIONS[PhaseAction.EAST_WEST]) == 2
assert len(PHASE_DIRECTIONS[PhaseAction.NORTH_SOUTH]) == 2
assert "initial_phase" in SumoTrafficEnv.__init__.__code__.co_varnames
print("Verified: Action 0=EAST_WEST, Action 1=NORTH_SOUTH, DQN output=2.")


In [ ]:
# torch is already preinstalled on Kaggle's GPU image - don't reinstall it
!pip install -q pydantic pydantic-settings structlog gymnasium numpy tqdm

In [ ]:
import os
import sumo

# eclipse-sumo bundles its own binaries + tools/ under the installed
# package directory - point SUMO_HOME there instead of a system path.
os.environ["SUMO_HOME"] = os.path.dirname(sumo.__file__)
print("SUMO_HOME =", os.environ["SUMO_HOME"])

!sumo --version

If `sumo --version` fails to print a version here, the `eclipse-sumo` wheel most likely didn't ship a binary for this exact platform. Fall back to the apt-get route as a last resort (slower, and known to segfault on some Kaggle images - see the note in cell 1):

```bash
!apt-get update -qq && apt-get install -y -qq sumo sumo-tools sumo-doc
```
```python
import os
os.environ["SUMO_HOME"] = "/usr/share/sumo"
```

## 2. Confirm GPU is visible to PyTorch, and libsumo is importable

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

import libsumo
print("libsumo importable OK - training will use the fast in-process backend")

## 3. Build the SUMO network and generate demand

In [ ]:
!python -m simulation.net.build_net
!python -m simulation.net.generate_routes --duration 3600 --seed 42

## 4. Quick pipeline smoke test (optional but recommended)

Runs 2 tiny episodes end-to-end before committing to a long training run - catches setup problems in seconds instead of hours.

In [ ]:
from rl.train.config import TrainingConfig
from rl.train.train import train

smoke_cfg = TrainingConfig(
    num_episodes=2,
    episode_duration_s=60,
    checkpoint_dir="/kaggle/working/smoke_checkpoints",
    min_replay_size=8,
    batch_size=4,
)
_ = train(smoke_cfg)
print("Smoke test OK")

## 5. Train the improved 2-phase policy

Recommended starting point: 1500 episodes, 3600s per episode, 10s minimum green, Double DQN target, and slower epsilon decay. Because each action now controls a whole road axis instead of one single approach, this matches real traffic-light operation much better and removes the collapse mode where the agent keeps choosing only `NORTH`.

The raw `dqn_best.pt` is still saved during training, but do **not** blindly deploy it. The next section evaluates candidate checkpoints and writes `dqn_eval_best.pt`, which is the file you should bring back to the local repo first.


In [ ]:
from rl.train.config import TrainingConfig
from rl.train.train import train

cfg = TrainingConfig(
    num_episodes=1500,
    episode_duration_s=3600,
    green_duration_s=10.0,
    checkpoint_dir="/kaggle/working/checkpoints",
    batch_size=128,
    min_replay_size=5000,
    replay_capacity=100_000,
    learning_rate=5e-5,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.02,
    epsilon_decay_episodes=1200,
    target_sync_every_episodes=5,
    checkpoint_every_episodes=50,
    train_every_n_steps=4,
    log_every_episodes=10,
)
agent = train(cfg)


## 6. Resuming (only needed if a session got cut off)

Kaggle sessions get killed after ~9-12 hours. `/kaggle/working/` output persists between sessions, so if training didn't finish, start a new session and continue from the last checkpoint instead of restarting from scratch.

In [ ]:
# from rl.train.config import TrainingConfig
# from rl.train.train import train
#
# resumed_cfg = TrainingConfig(
#     num_episodes=1000,
#     episode_duration_s=3600,
#     checkpoint_dir="/kaggle/working/checkpoints",
#     resume_from="/kaggle/working/checkpoints/dqn_final.pt",
# )
# agent = train(resumed_cfg)

## 7. Evaluate checkpoints and select the deployment model

This evaluates recent periodic checkpoints plus `dqn_best.pt` and `dqn_final.pt` on held-out seeds. The selected checkpoint is copied to `/kaggle/working/checkpoints/dqn_eval_best.pt`. Lower score is better; the score favors lower mean/final waiting time, lower queue length, and higher throughput.

In [ ]:
from pathlib import Path
import shutil

from benchmark.policies import DQNPolicy, FixedTimePolicy
from benchmark.run_episode import run_episode
from rl.agent.dqn_agent import DQNAgent
from rl.env.traffic_env import SumoTrafficEnv
from rl.train.checkpoint import load_checkpoint

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
EVAL_DURATION_S = 1200
EVAL_SEEDS = [101, 102, 103]

periodic = sorted(CHECKPOINT_DIR.glob("dqn_episode_*.pt"), key=lambda p: int(p.stem.split("_")[-1]))
candidate_paths = []
for path in [CHECKPOINT_DIR / "dqn_best.pt", CHECKPOINT_DIR / "dqn_final.pt", *periodic[-10:]]:
    if path.exists() and path not in candidate_paths:
        candidate_paths.append(path)


def mean(values):
    return sum(values) / len(values) if values else 0.0


def aggregate(metrics):
    dicts = [m.to_dict() for m in metrics]
    return {key: mean([d[key] for d in dicts]) for key in dicts[0]}


def score(metrics):
    # Lower is better. Throughput is subtracted because more arrivals are good.
    return (
        metrics["mean_waiting_time_s"]
        + 0.5 * metrics["final_waiting_time_s"]
        + 25.0 * metrics["mean_queue_length"]
        - 2.0 * metrics["arrived_vehicles"]
    )


env = SumoTrafficEnv(
    sumocfg_path="simulation/net/intersection.sumocfg",
    episode_duration_s=EVAL_DURATION_S,
    green_duration_s=10.0,
    backend="libsumo",
)
try:
    fixed_metrics = [run_episode(env, FixedTimePolicy(green_duration_s=20.0), seed=s) for s in EVAL_SEEDS]
finally:
    env.close()
fixed_agg = aggregate(fixed_metrics)
print("Fixed-time baseline:", fixed_agg)

results = []
for path in candidate_paths:
    agent = DQNAgent()
    trained_episode = load_checkpoint(path, agent)
    policy = DQNPolicy(agent, epsilon=0.0)
    env = SumoTrafficEnv(
        sumocfg_path="simulation/net/intersection.sumocfg",
        episode_duration_s=EVAL_DURATION_S,
        green_duration_s=10.0,
        backend="libsumo",
    )
    try:
        metrics = [run_episode(env, policy, seed=s) for s in EVAL_SEEDS]
    finally:
        env.close()
    agg = aggregate(metrics)
    agg_score = score(agg)
    results.append((agg_score, path, trained_episode, agg))
    print(
        f"{path.name:<22} ep={trained_episode:<5} score={agg_score:9.2f} "
        f"mean_wait={agg['mean_waiting_time_s']:8.2f} queue={agg['mean_queue_length']:6.2f} "
        f"arrived={agg['arrived_vehicles']:7.2f}"
    )

results.sort(key=lambda row: row[0])
best_score, best_path, best_episode, best_metrics = results[0]
selected_path = CHECKPOINT_DIR / "dqn_eval_best.pt"
shutil.copy2(best_path, selected_path)
print("\nSelected:", best_path.name, "episode", best_episode, "score", best_score)
print("Metrics:", best_metrics)
print("Wrote:", selected_path)


## 8. Download the selected model

Download `dqn_eval_best.pt` first. It is selected by deterministic evaluation across held-out seeds, so it is usually a better deployment candidate than the raw training `dqn_best.pt`. Keep `dqn_best.pt` and `dqn_final.pt` too if you want to compare locally.

In [ ]:
!ls -lh /kaggle/working/checkpoints


In [ ]:
import base64
from pathlib import Path
from IPython.display import HTML, display


def download_link(path, filename=None):
    filename = filename or path.split("/")[-1]
    with open(path, "rb") as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    return HTML(f'<a download="{filename}" href="data:application/octet-stream;base64,{b64}">Download {filename}</a>')


for path in [
    "/kaggle/working/checkpoints/dqn_eval_best.pt",
    "/kaggle/working/checkpoints/dqn_best.pt",
    "/kaggle/working/checkpoints/dqn_final.pt",
]:
    if Path(path).exists():
        display(download_link(path))
